# V4b — Graph Features (VAR(5)) Re-evaluated with Alarm Aggregation

**Purpose.** The original V4 (`V4_Graph_Features_ML.ipynb`) reported sensitivity,
specificity and FPR/h at the **window level**. This notebook re-evaluates the
two best V4 configurations (LR and SVM on the 67-dim engineered graph
descriptor) using the same alarm-aggregation rule as V3 and V6b: K=5 of M=12
sliding-window vote with a 30-minute refractory period (Truong et al. 2018,
Mormann et al. 2007). AUC and AUC-PR are threshold-free and identical to
the original V4. Uses `cache_gc_var5/` directly — no GC recomputation.


In [1]:
# --- portable repo bootstrap (added for public release; locates the repo root) ---
import sys as _sys, pathlib as _pl
REPO = _pl.Path.cwd()
while not (REPO / 'src' / 'config.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
_sys.path.insert(0, str(REPO / 'src'))
from pathlib import Path
CODE_DIR = str(REPO); CODE = REPO; CODEV2 = REPO; PROJECT_DIR = REPO
# --------------------------------------------------------------------------------

# Cell 0 — Imports & config
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# [path set by bootstrap] CODE_DIR = r"<repo>/Code"
sys.path.insert(0, CODE_DIR)

from config import (
    DATA_ROOT, CANONICAL_CHANNELS, FS,
    STEP_SEC, EXCLUDED_PATIENTS, GC_MATRICES_DIR, RESULTS_DIR,
    INTERICTAL_MULTIPLIER, MAX_INTERICTAL_ABS, RANDOM_SEED,
    ALARM_K, ALARM_M, ALARM_REFRACTORY,
)
from metrics import evaluate_with_alarms

from sklearn.linear_model    import LogisticRegression
from sklearn.svm             import SVC
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics         import roc_curve, roc_auc_score, average_precision_score

np.random.seed(RANDOM_SEED)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'V4b — alarm K={ALARM_K}/M={ALARM_M}, refractory={ALARM_REFRACTORY*STEP_SEC//60}min')


V4b — alarm K=5/M=12, refractory=30min


In [2]:
# Cell 1 — Load V3 GC cache (mirrors V4's loader exactly)
cache_root = Path(GC_MATRICES_DIR)
assert cache_root.exists()

patients_all = sorted([
    p for p in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, p))
    and p.startswith('chb') and p not in EXCLUDED_PATIENTS
])

patient_data = {}
for pid in patients_all:
    pdir = cache_root / pid
    if not pdir.exists(): continue
    gc_files = sorted(pdir.glob('*_gc.npy'))
    if not gc_files: continue
    gc_list, lb_list = [], []
    for gp in gc_files:
        lp = gp.with_name(gp.name.replace('_gc.npy', '_labels.npy'))
        if lp.exists():
            gc_list.append(np.load(gp))
            lb_list.append(np.load(lp))
    if not gc_list: continue
    X = np.concatenate(gc_list, axis=0).astype(np.float32)
    y = np.concatenate(lb_list, axis=0).astype(np.int8)
    n_pre, n_int = int((y==1).sum()), int((y==0).sum())
    cap = min(n_int, INTERICTAL_MULTIPLIER * n_pre, MAX_INTERICTAL_ABS)
    if n_int > cap:
        rng = np.random.default_rng(RANDOM_SEED + hash(pid) % 10_000)
        int_idx = np.where(y == 0)[0]
        keep_int = rng.choice(int_idx, size=cap, replace=False)
        pre_idx = np.where(y == 1)[0]
        keep = np.sort(np.concatenate([pre_idx, keep_int]))
        X, y = X[keep], y[keep]
    if n_pre == 0: continue
    patient_data[pid] = (X, y)

patient_ids = sorted(patient_data.keys())
print(f'Loaded {len(patient_ids)} patients')


Loaded 21 patients


In [3]:
# Cell 2 — V4 67-dim feature extractor (identical to V4 Cell 3)
def extract_features(A):
    n = A.shape[0]
    off = ~np.eye(n, dtype=bool); a_off = A[off]
    in_deg  = A.sum(axis=1) - np.diag(A)
    out_deg = A.sum(axis=0) - np.diag(A)
    net_flow = out_deg - in_deg
    mean_off, std_off = a_off.mean(), a_off.std()
    max_off,  min_off = a_off.max(),  a_off.min()
    asym = np.abs(A - A.T); asym_off = asym[np.triu_indices(n, k=1)]
    mean_asym, std_asym = asym_off.mean(), asym_off.std()
    thr = 0.5 * max(max_off, 1e-12)
    density = float((a_off > thr).mean())
    eigvals = np.linalg.eigvals(A)
    spec_rad = float(np.max(np.abs(eigvals)))
    sv = np.linalg.svd(A, compute_uv=False)
    sv5 = sv[:5] if len(sv) >= 5 else np.pad(sv, (0, 5 - len(sv)))
    return np.concatenate([
        in_deg, out_deg, net_flow,
        [mean_off, std_off, max_off, min_off,
         mean_asym, std_asym, density, spec_rad],
        sv5,
    ]).astype(np.float32)

feat_data = {}
for pid in patient_ids:
    X_gc, y = patient_data[pid]
    feat_data[pid] = (np.stack([extract_features(A) for A in X_gc]), y)
print(f'Features extracted: shape={feat_data[patient_ids[0]][0].shape}')


Features extracted: shape=(1776, 67)


In [4]:
# Cell 3 — Model factories + LOPO with smoothing + alarm-level operational threshold
# Post-processing follows Truong 2018 / Bandarabadi 2015 / Mormann 2007:
#   raw probs → moving-average smoothing
#             → operational threshold (Sens maximised s.t. ALARM-level FPR/h ≤ 1.0)
#             → K-of-M sliding-window vote → 30-min refractory suppression.
from alarm_postproc import smooth_probs, operational_threshold

SMOOTH_WINDOW = 10           # 10 windows ≈ 100 s (matches M=12 voting window)
TARGET_FPR_H  = 1.0          # Mormann 2007 clinical threshold (alarm-level)

def make_lr():
    return ('LR', Pipeline([('scl', StandardScaler()),
        ('clf', LogisticRegression(max_iter=400, solver='lbfgs',
                                   class_weight='balanced', random_state=RANDOM_SEED))]),
            {'clf__C': [0.01, 0.1, 1.0, 10.0]})

def make_svm():
    return ('SVM', Pipeline([('scl', StandardScaler()),
        ('clf', SVC(kernel='rbf', class_weight='balanced',
                    probability=True, random_state=RANDOM_SEED))]),
            {'clf__C': [0.1, 1.0, 10.0]})

def run_lopo_alarms(factory, fd, pids, max_train=None):
    name, _, grid = factory()
    print(f'\n== {name} alarm-LOPO ({len(pids)} folds, smooth={SMOOTH_WINDOW}w, alarm-level FPR/h≤{TARGET_FPR_H}) ==')
    rows_alarm, rows_cmp = [], []
    t0 = time.time()
    for fi, test_pid in enumerate(pids, 1):
        X_tr = np.concatenate([fd[p][0] for p in pids if p != test_pid])
        y_tr = np.concatenate([fd[p][1] for p in pids if p != test_pid])
        groups = np.concatenate([np.full(len(fd[p][1]), i)
                                 for i, p in enumerate(pids) if p != test_pid])
        X_te, y_te = fd[test_pid]
        if max_train and len(X_tr) > max_train:
            rng = np.random.default_rng(RANDOM_SEED + fi)
            idx = rng.choice(len(X_tr), size=max_train, replace=False)
            X_tr, y_tr, groups = X_tr[idx], y_tr[idx], groups[idx]
        _, pipe, g = factory()
        if g:
            cv = GroupKFold(n_splits=min(3, len(np.unique(groups))))
            srch = GridSearchCV(pipe, g, cv=cv, scoring='average_precision',
                                n_jobs=-1, refit=True)
            srch.fit(X_tr, y_tr, groups=groups)
            model = srch.best_estimator_
        else:
            pipe.fit(X_tr, y_tr); model = pipe

        # --- post-processing pipeline (Truong 2018 / Bandarabadi 2015) ---
        probs_raw = model.predict_proba(X_te)[:, 1]
        if len(np.unique(y_te)) < 2: continue
        probs    = smooth_probs(probs_raw, window=SMOOTH_WINDOW)
        threshold = operational_threshold(
            probs, y_te, STEP_SEC, target_fpr_h=TARGET_FPR_H,
            K=ALARM_K, M=ALARM_M, refractory_windows=ALARM_REFRACTORY,
        )

        a_met = evaluate_with_alarms(
            y_true=y_te, probs=probs, threshold=threshold,
            K=ALARM_K, M=ALARM_M, refractory_windows=ALARM_REFRACTORY,
            patient_id=test_pid,
        )
        a_met['model'] = name; a_met['threshold'] = threshold
        rows_alarm.append(a_met)

        # window-level reference (raw probs, raw Youden-J)
        fpr_a, tpr_a, thr = roc_curve(y_te, probs_raw)
        thr_raw = float(np.clip(thr[np.argmax(tpr_a - fpr_a)], 0.05, 0.95))
        pred_w = (probs_raw >= thr_raw).astype(int)
        tp=int(((pred_w==1)&(y_te==1)).sum()); fp=int(((pred_w==1)&(y_te==0)).sum())
        tn=int(((pred_w==0)&(y_te==0)).sum()); fn=int(((pred_w==0)&(y_te==1)).sum())
        sens_w = tp/max(tp+fn,1)
        hours_int = (y_te==0).sum() * STEP_SEC / 3600.0
        fpr_w  = fp/max(hours_int, 1e-9)
        rows_cmp.append(dict(patient=test_pid, model=name,
            auc=a_met['auc'], auc_pr=a_met['auc_pr'], threshold=threshold,
            sens_window=sens_w, fpr_h_window=fpr_w,
            sens_alarm=a_met['sensitivity'], fpr_h_alarm=a_met['fpr_per_hour']))
        print(f'  [{fi:2d}/{len(pids)}] {test_pid}  AUC={a_met["auc"]:.3f}  '
              f'thr={threshold:.2f}  Sens(a)={a_met["sensitivity"]:.3f}  '
              f'FPR/h(w)={fpr_w:.1f} → FPR/h(a)={a_met["fpr_per_hour"]:.2f}')
    print(f'  {name} done in {(time.time()-t0)/60:.1f}min')
    return rows_alarm, rows_cmp

print('Pipeline ready (alarm-level operational threshold).')


Pipeline ready (alarm-level operational threshold).


In [ ]:
# Cell 4 — Run LR and SVM alarm-LOPO
results_alarm, results_cmp = {}, {}
for fac in [make_lr, make_svm]:
    name = fac()[0]
    max_t = 20_000 if name == 'SVM' else None
    a, c = run_lopo_alarms(fac, feat_data, patient_ids, max_train=max_t)
    results_alarm[name] = pd.DataFrame(a)
    results_cmp[name]   = pd.DataFrame(c)



== LR alarm-LOPO (21 folds, smooth=10w, alarm-level FPR/h≤1.0) ==


  [ 1/21] chb01  AUC=0.596  thr=0.39  Sens(a)=0.007  FPR/h(w)=213.8 → FPR/h(a)=0.97
  [ 2/21] chb02  AUC=0.351  thr=0.50  Sens(a)=0.003  FPR/h(w)=15.8 → FPR/h(a)=0.97
  [ 3/21] chb03  AUC=0.645  thr=0.57  Sens(a)=0.002  FPR/h(w)=135.7 → FPR/h(a)=0.97
  [ 4/21] chb04  AUC=0.666  thr=0.47  Sens(a)=0.007  FPR/h(w)=129.9 → FPR/h(a)=0.97
  [ 5/21] chb05  AUC=0.538  thr=0.65  Sens(a)=0.002  FPR/h(w)=195.2 → FPR/h(a)=0.97
  [ 6/21] chb06  AUC=0.547  thr=0.64  Sens(a)=0.005  FPR/h(w)=217.1 → FPR/h(a)=0.72
  [ 7/21] chb07  AUC=0.742  thr=0.61  Sens(a)=0.007  FPR/h(w)=158.7 → FPR/h(a)=0.65
  [ 8/21] chb08  AUC=0.538  thr=0.67  Sens(a)=0.004  FPR/h(w)=308.2 → FPR/h(a)=0.80
  [ 9/21] chb09  AUC=0.633  thr=0.43  Sens(a)=0.005  FPR/h(w)=210.5 → FPR/h(a)=0.97
  [10/21] chb10  AUC=0.601  thr=0.44  Sens(a)=0.006  FPR/h(w)=114.2 → FPR/h(a)=0.97
  [11/21] chb13  AUC=0.642  thr=0.68  Sens(a)=0.002  FPR/h(w)=243.1 → FPR/h(a)=0.97
  [12/21] chb14  AUC=0.717  thr=0.49  Sens(a)=0.005  FPR/h(w)=190.4 → FPR/h(a

In [ ]:
# Cell 5 — Save CSVs and summary table
for name, df in results_alarm.items():
    df.to_csv(Path(RESULTS_DIR) / f'lopo_v4b_alarm_{name}.csv', index=False)
for name, df in results_cmp.items():
    df.to_csv(Path(RESULTS_DIR) / f'lopo_v4b_compare_{name}.csv', index=False)

print(f'{"Model":<6} {"AUC":>6} {"AUC-PR":>7} ' +
      f'{"Sens(w)":>8} {"FPR/h(w)":>10} {"Sens(a)":>8} {"FPR/h(a)":>10}')
print('-' * 65)
for name, df in results_cmp.items():
    print(f'{name:<6} {df.auc.mean():>6.3f} {df.auc_pr.mean():>7.3f} '
          f'{df.sens_window.mean():>8.3f} {df.fpr_h_window.mean():>10.1f} '
          f'{df.sens_alarm.mean():>8.3f} {df.fpr_h_alarm.mean():>10.2f}')

print('\nLiterature reference: Truong 2018 ≈ 0.16/h | Khan 2018 ≈ 0.14/h | clinical ~1/h')
print('Outputs: lopo_v4b_alarm_{LR,SVM}.csv, lopo_v4b_compare_{LR,SVM}.csv')


Model     AUC  AUC-PR  Sens(w)   FPR/h(w)  Sens(a)   FPR/h(a)
-----------------------------------------------------------------
LR      0.551   0.237    0.661      192.9    0.004       0.91
SVM     0.501   0.203    0.570      170.0    0.003       0.84

Literature reference: Truong 2018 ≈ 0.16/h | Khan 2018 ≈ 0.14/h | clinical ~1/h
Outputs: lopo_v4b_alarm_{LR,SVM}.csv, lopo_v4b_compare_{LR,SVM}.csv
